# 🛍️ Customer Segmentation using K-Means Clustering
### Domain: Retail Analytics & Unsupervised Machine Learning

In this project, we apply **K-Means Clustering** to the classic **Mall Customers Dataset** to segment shoppers based on their **Annual Income** and **Spending Score**.

---
### Pipeline Overview:
1. **Phase 1: Setup & Exploratory Data Analysis (EDA)**
2. **Phase 2: Finding Optimal K (The Elbow Method)**
3. **Phase 3: Model Training with Optimal K ($K = 5$)**
4. **Phase 4: Cluster Visualization & Business Persona Profiling**

## Phase 1: Setup & Data Exploration
Let us import the necessary libraries, load the dataset, check for missing values, and inspect initial distributions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
import os

# Visual formatting
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 11})

# Handle dataset location across Kaggle and local environments
dataset_filename = "Mall_Customers.csv"
kaggle_path = f"/kaggle/input/customer-segmentation-tutorial-in-python/{dataset_filename}"
filepath = kaggle_path if os.path.exists(kaggle_path) else dataset_filename

df = pd.read_csv(filepath)
print(f"Dataset loaded successfully with shape: {df.shape}")
df.head()

In [ ]:
# Dataset information and null check
print("--- Data Types & Non-Null Counts ---")
print(df.info())

print("\n--- Missing Values Audit ---")
print(df.isnull().sum())

print("\n--- Summary Statistics ---")
display(df.describe())

In [ ]:
# Initial Scatter Plot: Annual Income vs. Spending Score
plt.figure(figsize=(9, 6), dpi=120)
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    color="#1f77b4",
    s=80,
    alpha=0.85,
    edgecolor="w"
)
plt.title("Initial Exploration: Annual Income vs. Spending Score", fontsize=14, weight="bold", pad=15)
plt.xlabel("Annual Income (k$)", fontsize=12)
plt.ylabel("Spending Score (1-100)", fontsize=12)
plt.show()

## Phase 2: Finding Optimal K (The Elbow Method)
We extract **Annual Income** and **Spending Score** as our clustering features.

We compute the **Within-Cluster Sum of Squares (WCSS / Inertia)** for $K = 1$ to $10$:
$$\text{WCSS} = \sum_{k=1}^{K} \sum_{x \in C_k} \| x - \mu_k \|^2$$

The "Elbow Point" marks the number of clusters where adding more clusters yields diminishing returns in error reduction.

In [ ]:
# Extract features: Annual Income (col 3) & Spending Score (col 4)
X = df.iloc[:, [3, 4]].values

wcss = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, init="k-means++", random_state=42, n_init=10)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)
    print(f"K = {k:2d} -> WCSS (Inertia): {kmeans.inertia_:10.2f}")

# Plot the Elbow Curve
plt.figure(figsize=(9, 5.5), dpi=120)
plt.plot(k_range, wcss, marker='o', markersize=8, color="#2b5c8f", linewidth=2.5, label="WCSS")

optimal_k = 5
plt.axvline(x=optimal_k, color="#d9534f", linestyle="--", linewidth=1.8, label=f"Elbow Point (K = {optimal_k})")
plt.scatter(optimal_k, wcss[optimal_k - 1], color="#d9534f", s=130, zorder=5)

plt.title("The Elbow Method for Selecting Optimal Clusters", fontsize=14, weight="bold", pad=15)
plt.xlabel("Number of Clusters (K)", fontsize=12)
plt.ylabel("WCSS (Inertia)", fontsize=12)
plt.xticks(k_range)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(frameon=True, facecolor="white", loc="upper right")
plt.show()

## Phase 3: Model Training (K-Means with $K = 5$)
We now train our final K-Means model using $K = 5$ clusters and `k-means++` initialization.

In [ ]:
# Initialize and fit model
kmeans = KMeans(n_clusters=5, init="k-means++", random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X)
centroids = kmeans.cluster_centers_

print("Model training complete. Centroid coordinates (Income, Spending Score):")
for i, center in enumerate(centroids):
    print(f"  Cluster {i}: Annual Income = ${center[0]:5.2f}k, Spending Score = {center[1]:5.2f}")

## Phase 4: Visualization & Business Persona Profiling
We assign intuitive business personas to each cluster based on their centroid position:

In [ ]:
# Define Persona Mapping
persona_map = {}
for i, center in enumerate(centroids):
    inc, spd = center[0], center[1]
    if inc > 65 and spd > 60:
        persona = "VIP / Target Stars (High Income, High Spend)"
    elif inc > 65 and spd < 40:
        persona = "Careful / Conservative (High Income, Low Spend)"
    elif inc < 45 and spd > 60:
        persona = "Impulsive / Trendsetters (Low Income, High Spend)"
    elif inc < 45 and spd < 40:
        persona = "Budget Shoppers (Low Income, Low Spend)"
    else:
        persona = "Middle Ground (Moderate Income, Moderate Spend)"
    persona_map[i] = persona

df["Segment_Name"] = df["Cluster"].map(persona_map)

# Final Clustered Scatter Plot
palette = ["#e74c3c", "#3498db", "#2ecc71", "#9b59b6", "#f39c12"]
plt.figure(figsize=(11, 7), dpi=130)

for idx, cluster_id in enumerate(sorted(df["Cluster"].unique())):
    subset = df[df["Cluster"] == cluster_id]
    plt.scatter(
        subset["Annual Income (k$)"],
        subset["Spending Score (1-100)"],
        s=85,
        color=palette[idx],
        alpha=0.85,
        edgecolor="w",
        label=f"Cluster {cluster_id}: {persona_map[cluster_id]}"
    )

# Overlay Centroids
plt.scatter(
    centroids[:, 0],
    centroids[:, 1],
    s=260,
    color="#111111",
    marker="X",
    edgecolor="#ffffff",
    linewidth=1.8,
    zorder=10,
    label="Centroids"
)

plt.title("Customer Segmentation: 5 Distinct Customer Groups", weight="bold", pad=15)
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0.)
plt.tight_layout()
plt.show()

In [ ]:
# Aggregated Demographic & Behavioral Summary
summary = df.groupby(["Cluster", "Segment_Name"]).agg(
    Customer_Count=("CustomerID", "count"),
    Avg_Age=("Age", "mean"),
    Avg_Income_k=("Annual Income (k$)", "mean"),
    Avg_Spending_Score=("Spending Score (1-100)", "mean"),
    Female_Percentage=("Genre", lambda g: round((g == "Female").mean() * 100, 1))
).reset_index()

display(summary)